Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

 Messages are objects that contain:

Role - Identifies the message type (e.g. system, user).

Content - Represents the actual content of the message (like text, images, audio, documents, etc.)

Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

1) Text Prompts
2) Message Prompts

Text Prompts

Text prompts are strings - ideal for straightforward generation tasks where vou don't need to retain conversation history.


In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

model.invoke("what are birds?") #Text prompt that will generate AIMessage

AIMessage(content='<think>\nOkay, so the user asked "what are birds?" I need to start by breaking down the question. First, birds are a class of animals, right? So maybe start by defining them as warm-blooded vertebrates. Then, I should mention their key characteristics. Feathers come to mind immediately. All birds have feathers, which are unique to them. Also, they have beaks or bills, which they use for eating. They lay eggs, which is another important point. Oh, and most birds can fly, but not all. For example, penguins and ostriches can\'t fly. I should note that exception.\n\nNext, classification. Birds belong to the class Aves. They evolved from dinosaurs, specifically theropod dinosaurs. That\'s a cool fact. Maybe include something about the fossil record, like Archaeopteryx as a transitional species. Then, their anatomy: hollow bones for flight, strong muscles, especially in the chest for flapping. The respiratory system is also unique with air sacs. \n\nThey have different typ

Message Prompts

Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types:
System message - Tells the model how to behave and provide context for interactions(These define the AI's behavior, persona, and overall instructions for the interaction.)

Human message - These represent the user's questions, requests, or input provided to the model.

Al message - Responses generated by the model, including text content, tool call requests, and metadata

Tool message - These contain the results or outputs returned from external tools 

System Message

A SystemMessage represent an initial set of instructions that primes the model's behavior. You can use a system message to set the tone, define the model's role, and establish guidelines for responses.

Human message

A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

Al Message

An AlMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

Tool Message

For models that support tool calling, Al messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [2]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
#more detailed the system message to LLM, the better the response.
messages=[
    SystemMessage("You are a poet"),
    HumanMessage("Write a poem on parents")
]
response=model.invoke(messages)
response.content


'<think>\nOkay, I need to write a poem about parents. Let me start by thinking about what aspects of parents I can highlight. Parents are often seen as caregivers, providers, and sources of love and support. Maybe I can touch on their roles in different stages of a child\'s life.\n\nFirst, the structure. A traditional poem with stanzas and rhyme scheme? Maybe four-line stanzas with an ABAB rhyme scheme. That\'s common and flows well. Let me try that.\n\nNow, imagery. Nature metaphors often work well. Maybe compare parents to trees, providing shelter and stability. Or maybe a lighthouse, guiding through storms. Also, the idea of time—how parents age as their children grow. Maybe contrast the child\'s growth with the parents\' enduring presence.\n\nThemes to include: unconditional love, sacrifices made, guidance, support through hardships, and gratitude. Also, the idea that even when children stray or face challenges, parents remain a constant.\n\nPossible lines: Start with a child\'s pe

In [6]:
## Message Metadata

human_msg=HumanMessage(
content="Hello!",
name="alice", # Optional: identify different users
id="msg_123", #Optional: unique identifier for tracing
)

response=model.invoke([human_msg])
response
response.usage_metadata


{'input_tokens': 10, 'output_tokens': 114, 'total_tokens': 124}

In [7]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

#After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message=AIMessage(
content=[],
tool_calls=[
    {
"name": "get_weather",
"args": {"location": "San Francisco"},
"id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
content=weather_result,
tool_call_id="call_123" #Must match the call ID
)

#Continue conversation
messages = [
HumanMessage("What's the weather in San Francisco?"),
ai_message, #Model's tool call
tool_message, #Tool execution result
]
response=model.invoke(messages) # Model processes the result